# GRASP One-Shot Designer (Colab)

Design **Golden Gate oligos** for a binder that recognizes **one target RNA**.

No combinatorial library modules. Flow: RNA → protein → free cut sites → optimized oligos.

**Colab tip:** each settings cell uses **Forms**. Use the cell ⋮ menu → **Form → Hide code** (or open the notebook — titles use `{display-mode: "form"}` so code stays hidden by default).


In [ ]:
#@title 0 · Install { display-mode: "form" }
#@markdown Prefer **PyPI** once published. Use **Private GitHub** with a PAT for the private repo.

install_from = "Private GitHub" #@param ["PyPI", "Private GitHub", "Local editable"]
GITHUB_TOKEN = "" #@param {type:"string"}
REPO_SLUG = "JustABiologist/grasp-library-designer" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
PYPI_VERSION = "" #@param {type:"string"}
#@markdown Leave `PYPI_VERSION` empty for latest. Example: `0.1.0`

import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

mode = str(install_from)

if mode == "PyPI":
    spec = "grasp-library-designer"
    if str(PYPI_VERSION).strip():
        spec = f"{spec}=={str(PYPI_VERSION).strip()}"
    _pip(spec)
    print("Installed", spec)

elif mode == "Private GitHub":
    if IN_COLAB:
        repo_dir = Path("/content/grasp-library-designer")
        if not (repo_dir / "grasp_library").is_dir():
            if not REPO_SLUG or "OWNER/" in REPO_SLUG:
                raise ValueError("Set REPO_SLUG to your GitHub owner/repo (private).")
            auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN.strip() else ""
            url = f"https://{auth}github.com/{REPO_SLUG}.git"
            print(f"Cloning github.com/{REPO_SLUG} @ {BRANCH} …")
            subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(repo_dir)])
        os.chdir(repo_dir)
        sys.path.insert(0, str(repo_dir))
        _pip("-e", ".")
        print("Installed from", repo_dir)
    else:
        root = Path.cwd()
        if not (root / "grasp_library").is_dir():
            for p in [root, *root.parents]:
                if (p / "grasp_library").is_dir():
                    os.chdir(p)
                    sys.path.insert(0, str(p))
                    break
        _pip("-e", ".")
        print("Using local package at", Path.cwd())

else:  # Local editable
    root = Path.cwd()
    if not (root / "grasp_library").is_dir():
        for p in [root, *root.parents]:
            if (p / "grasp_library").is_dir():
                os.chdir(p)
                sys.path.insert(0, str(p))
                break
    _pip("-e", ".")
    print("Editable install at", Path.cwd())

print("Ready.")


In [ ]:
#@title 1 · Settings { display-mode: "form" }
#@markdown Choose organism, synthesis, ligation, and target RNA — then run this cell.
#@markdown **Codon tables:** browse [Kazusa CUTG](https://www.kazusa.or.jp/codon/) (search → copy the `species=` accession from the URL). Built-ins cover common hosts; otherwise choose *Fetch from Kazusa* or *Upload your own*.

target_rna = "UUACACGUG" #@param {type:"string"}
organism = "Escherichia coli (Kazusa)" #@param ["Escherichia coli (Kazusa)", "Saccharomyces cerevisiae (Kazusa)", "Homo sapiens (Kazusa)", "Euglena gracilis nuclear (Kazusa)", "Chlamydomonas reinhardtii nuclear (Kazusa)", "Chlamydomonas reinhardtii chloroplast (Kazusa)", "Fetch from Kazusa by species ID", "Upload your own codon table"]
kazusa_species_id = "" #@param {type:"string"}
genetic_code = 1 #@param {type:"integer"}
synthesis_vendor = "Twist \u00b7 Standard gene guidelines" #@param ["Twist \u00b7 Express / Low complexity", "Twist \u00b7 Standard gene guidelines", "Twist \u00b7 Complex Genes tolerant", "IDT \u00b7 gBlocks / eBlocks conservative", "Generic \u00b7 conservative (default)"]
assembly_enzyme = "GRASP default \u00b7 BsaI + BpiI + BsmBI" #@param ["GRASP default \u00b7 BsaI + BpiI + BsmBI", "BsaI (GGTCTC)", "BpiI / BbsI (GAAGAC)", "BsmBI / Esp3I (CGTCTC)", "None (no enzyme filter)"]
ligation_table = "T4 \u00b7 18 h \u00b7 25 \u00b0C (Potapov)" #@param ["T4 \u00b7 18 h \u00b7 25 \u00b0C (Potapov)", "T4 \u00b7 18 h \u00b7 37 \u00b0C (Potapov)", "T4 \u00b7 1 h \u00b7 25 \u00b0C (Potapov)", "BsaI-HFv2 \u00b7 constant 37 \u00b0C", "BsmBI-v2 \u00b7 constant 42 \u00b0C"]
optimize_depth = 2000 #@param {type:"integer"}
n_fragments = 0 #@param {type:"integer"}
#@markdown `n_fragments = 0` means auto (from oligo length limits).

from pathlib import Path
from grasp_library import build_default_config, materialize_project
from grasp_library.colab_forms import apply_form_settings
from grasp_library import notebook_ui as ui

PROJECT_DIR = materialize_project()
INPUT_DIR = PROJECT_DIR / "input"
OUTPUT_ROOT = PROJECT_DIR / "output" / "oneshot"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CONFIG = build_default_config(INPUT_DIR)
CONFIG["project_name"] = "GRASP_oneshot_colab"

ui.kazusa_codon_reminder()

applied = apply_form_settings(
    CONFIG,
    organism=organism,
    genetic_code=int(genetic_code),
    target_rna=target_rna,
    synthesis_vendor=synthesis_vendor,
    assembly_enzyme=assembly_enzyme,
    ligation_table=ligation_table,
    optimize_depth=int(optimize_depth),
    n_fragments=int(n_fragments) if int(n_fragments) > 0 else None,
    overhang_redesign=False,  # unused in protein-first oneshot
    kazusa_species_id=kazusa_species_id,
)
CONFIG = applied["config"]
CODON_DATA = applied["codon_data"]
ONESHOT = None
ORGANISM_LABEL = applied.get("meta", {}).get("organism", organism)

ui.status(
    f"Target <b>{CONFIG['target_rna']}</b> · organism <b>{ORGANISM_LABEL}</b> · "
    f"depth <b>{CONFIG['optimizer']['iterations_per_part']:,}</b> · "
    f"vendor <b>{CONFIG['synthesis_vendor']}</b>"
)


In [ ]:
#@title 2 · Preview binder protein { display-mode: "form" }

from grasp_library import describe_binder, suggest_fragment_count
from grasp_library import notebook_ui as ui

info = describe_binder(CONFIG["target_rna"])
n_auto = suggest_fragment_count(info["cds_length"])
ui.status(
    f"<b>{info['target_rna']}</b> · PPR <code>{info['ppr_code']}</code> · "
    f"<b>{info['aa_length']}</b> aa · <b>{info['cds_length']}</b> nt · "
    f"auto fragments ≈ <b>{n_auto}</b>"
)
print(info["aa_sequence"])


In [ ]:
#@title 3 · Design oligos { display-mode: "form" }
#@markdown Re-run after changing settings. Downloads appear in the files panel under `grasp_library_project/output/oneshot/`.

RUN_ONESHOT = True #@param {type:"boolean"}
SEED = 42 #@param {type:"integer"}

import random
import numpy as np
import pandas as pd
from IPython.display import display
from grasp_library import LigationFidelityCalculator, run_oneshot_design, sanitize_rna_name
from grasp_library import notebook_ui as ui

random.seed(int(SEED))
np.random.seed(int(SEED))

ONESHOT = None
if not RUN_ONESHOT:
    ui.note("RUN_ONESHOT is off.")
elif not CODON_DATA:
    ui.note("Run the Settings cell first.")
else:
    rna = sanitize_rna_name(CONFIG["target_rna"])
    out_dir = OUTPUT_ROOT / rna
    lig = CONFIG["ligation"]
    fidelity = LigationFidelityCalculator(
        temperature=lig["temperature"],
        hours=lig["hours"],
        ligation_table=lig.get("ligation_table"),
        min_efficiency=lig.get("min_efficiency", 0.25),
        min_fidelity=lig.get("min_fidelity", 0.9),
    )
    n_frag = CONFIG.get("oneshot_n_fragments")
    ONESHOT = run_oneshot_design(
        target_rna=CONFIG["target_rna"],
        codon_data=CODON_DATA,
        config=CONFIG,
        output_dir=out_dir,
        seed=int(SEED),
        n_fragments=n_frag,
        fidelity=fidelity,
        log=print,
    )
    display(ONESHOT["gga_plan"][
        ["fragment_id", "aa_start_0based", "aa_end_0based", "oh5", "oh3", "ligation_fidelity_set"]
    ])
    cols = [c for c in [
        "fragment_id", "assembly_order", "oligo_length", "oligo_gc",
        "qc_passed", "oligo_sequence_5to3",
    ] if c in ONESHOT["oligos"].columns]
    display(ONESHOT["oligos"][cols])
    asm = ONESHOT["assembled"]
    ui.status(
        f"Translation verified: <b>{asm['translation_verified']}</b> · "
        f"ligation fidelity <b>{asm['ligation_fidelity']:.4f}</b> · "
        f"oligos → <code>{ONESHOT['oligo_csv']}</code>"
    )


In [ ]:
#@title 4 · Export Excel { display-mode: "form" }

import pandas as pd
from grasp_library import notebook_ui as ui

if ONESHOT is None:
    ui.note("Run Design first.")
else:
    out = ONESHOT["output_dir"]
    xlsx = out / f"oneshot_{ONESHOT['target_rna']}.xlsx"
    with pd.ExcelWriter(xlsx) as writer:
        pd.DataFrame([ONESHOT["binder"]]).to_excel(writer, sheet_name="binder", index=False)
        ONESHOT["gga_plan"].to_excel(writer, sheet_name="gga_plan", index=False)
        ONESHOT["oligos"].to_excel(writer, sheet_name="oligos", index=False)
        pd.DataFrame([ONESHOT["summary"]]).to_excel(writer, sheet_name="summary", index=False)
    ui.status(f"Wrote <code>{xlsx}</code>")
    if "google.colab" in __import__("sys").modules:
        from google.colab import files
        files.download(str(xlsx))
    for p in sorted(out.glob("*")):
        if p.is_file():
            print(p.name)


## Notes

| Step | What happens |
|---|---|
| RNA → protein | PPR code (A→TN, C→NN, G→TD, U→ND) into GRASP repeat scaffold |
| Optimize | Full CDS codon + synthesis fitness for the chosen organism/vendor |
| Cuts | Codon-aligned; overhangs chosen for Potapov ligation fidelity |
| Export | Flanked oligos for Golden Gate |

For the **42-module combinatorial library**, open `grasp_library_designer.ipynb`.

Hidden form code is only a presentation setting — the source remains in the notebook file.
